# 🧪 Track A Challenge Evaluator

This notebook provides the following evaluations:

- **Validation:** Segmentation-distance evaluation between simulated and translated images.
- **Realism:** Distribution-level and single-image metrics comparing real images with translated images.
- **Behavioural realism:** Agreement of ISU predictions for simulated and translated images, evaluated with the selected failure-detection model.
- **Failure detection and diversity:** Diversity of failures detected by ISU when processing the images, evaluated with the selected failure-detection model.
- **Efficiency:** Translation time measured on specified hardware.

The evaluator writes raw and per-contestant reports to `track_a_evaluation_output`. Cross-contestant Pareto ranking is handled separately in the organizer-only ranking notebook.

In [10]:
import gc
import importlib
import json
import os
import sys
from pathlib import Path

# The kernel may start in the repository root, track_a/, or another subfolder such as track_b/.
# Find track_a/ from the current folder or its parents and work from there, so the local
# modules, submissions/ and the output folder resolve.
def find_track_a_dir(start):
    for folder in (start, *start.parents):
        for candidate in (folder, folder / 'track_a'):
            if (candidate / 'track_a_metrics.py').exists():
                return candidate
    raise FileNotFoundError(f'Could not find track_a/ from {start}. Open the notebook from the repository.')


TRACK_A_DIR = find_track_a_dir(Path.cwd().resolve())
os.chdir(TRACK_A_DIR)
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))

# venv/bin/python is the Linux/macOS layout; venv/Scripts/python.exe is the Windows layout.
venv_candidates = tuple(
    TRACK_A_DIR.parent / 'venv' / relative
    for relative in (Path('bin') / 'python', Path('Scripts') / 'python.exe')
)
VENV_PYTHON = next((path.resolve() for path in venv_candidates if path.exists()), None)
ACTIVE_PYTHON = Path(sys.executable).resolve()
if VENV_PYTHON is None or ACTIVE_PYTHON != VENV_PYTHON:
    expected = str(VENV_PYTHON) if VENV_PYTHON else 'the repository venv python'
    raise RuntimeError(
        f'Select the Track A venv kernel and restart the notebook. '
        f'Expected: {expected}; active: {ACTIVE_PYTHON}'
    )

import track_a_metrics
importlib.reload(track_a_metrics)
from image_metrics_utils import (
    MoondreamAnalyzer,
    Qwen25VLAnalyzer,
    TransformersSamSegmenter,
    save_report,
)
from track_a_metrics import *

OUTPUT = Path('track_a_evaluation_output')
OUTPUT.mkdir(exist_ok=True)
contestants = discover_contestants()
reports = {name: {'contestant': name} for name in contestants}
print(f'Working directory: {TRACK_A_DIR}')
print(f'Found {len(contestants)} contestant(s): {list(contestants)}')

Working directory: C:\Users\levia\Documents\testing\ISU-Challenge\Starterkit-ICSE-2027\track_a
Found 1 contestant(s): ['your_team_name']


## 🧩 SAM segmentation distance

This cell measures the structural distance between each simulated source image and its generated translation. A class-agnostic Segment Anything Model (SAM) creates object partitions for both images, and the evaluator compares those partitions without using semantic class names.

The comparison is paired by image stem:

- **Simulated vs. generated:** checks whether the translation preserves the source scene's object boundaries and spatial structure. This runs on all scenes.
- Real reference images are **not** used in this SAM calculation because they are not pixel-aligned with the simulated renders. Their visual similarity is evaluated in the later realism section.

The report contains partition-agreement metrics including boundary F1, symmetric region covering, matched IoU, adjusted Rand index (ARI), and normalized mutual information (NMI). These metrics are reported together with `semantic_segmentation_score = 1 - matched_iou`, which acts as the segmentation distance: lower values indicate closer structural agreement, while the agreement metrics are generally better when higher. The result is a continuous diagnostic; no pass/fail threshold is applied.

In [11]:
sam = TransformersSamSegmenter('facebook/sam-vit-base')
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: SAM distance for {name}')
    reports[name]['sam'] = evaluate_sam(folders, OUTPUT / name / 'sam', sam)
    save_report(reports[name], OUTPUT / f'{name}.json')
del sam
gc.collect()
print('SAM distance complete.')

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


Contestant 1/1: SAM distance for your_team_name
mapping: {'simulated': {'sample_00000': WindowsPath('submissions/your_team_name/simulated/sample_00000_sim.png'), 'sample_00001': WindowsPath('submissions/your_team_name/simulated/sample_00001_sim.png'), 'sample_00002': WindowsPath('submissions/your_team_name/simulated/sample_00002_sim.png'), 'sample_00003': WindowsPath('submissions/your_team_name/simulated/sample_00003_sim.png'), 'sample_00004': WindowsPath('submissions/your_team_name/simulated/sample_00004_sim.png'), 'sample_00005': WindowsPath('submissions/your_team_name/simulated/sample_00005_sim.png'), 'sample_00006': WindowsPath('submissions/your_team_name/simulated/sample_00006_sim.png'), 'sample_00007': WindowsPath('submissions/your_team_name/simulated/sample_00007_sim.png'), 'sample_00008': WindowsPath('submissions/your_team_name/simulated/sample_00008_sim.png')}, 'generated': {'sample_00000': WindowsPath('submissions/your_team_name/generated/sample_00000.png'), 'sample_00001': W

## 1. 📊 Realism evaluation

This section evaluates visual similarity and distribution-level realism using public image metrics. The evaluator reports results separately for generated-versus-real and simulated-versus-real comparisons, so the generated images can be compared with the original baseline.

The real images are read from the submission's `reference/` folder, which the participant starter copies from `data/real/`. They are named `<stem>_real.jpg` (e.g. `sample_00059_real.jpg`), and only the first 60 scenes (`sample_00000` to `sample_00059`) have one. **All metrics in this section are therefore computed only on those 60 scenes**, including the distribution metrics.

It applies two groups of metrics:

- **Paired image metrics:** exact pixel match, correlation coefficient, histogram intersection, LBP-histogram similarity, PSNR, SSIM, normalized mutual information, fractal-dimension similarity, KL divergence, MSE, VGG perceptual distance, GLCM texture distance, and Wasserstein distance. These compare each scene with its real reference image and assess pixel, structure, texture, histogram, and perceptual differences.
- **Distribution metrics:** Fréchet Inception Distance (FID), Kernel Inception Distance (KID), and Inception Score. These compare image distributions rather than individual pixels: lower FID and KID indicate closer distributions, while higher Inception Score is preferred.

For paired metrics, similarity measures such as SSIM and PSNR are generally better when higher, while distance or error measures such as MSE, KL divergence, Wasserstein distance, and VGG perceptual distance are better when lower. The raw values are reported for analysis; the later ranking step determines how they contribute to comparisons across participants.

In [12]:
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: public realism for scenes with a real reference image in {name}')
    reports[name]['realism_raw'] = evaluate_public_realism(folders)
    save_report(reports[name], OUTPUT / f'{name}.json')
print('Public realism metrics complete.')


Contestant 1/1: public realism for scenes with a real reference image in your_team_name
Scenes with a real reference image: 60 (generated: 9, simulated: 9)
mapping: {'generated': {'sample_00000': WindowsPath('submissions/your_team_name/generated/sample_00000.png'), 'sample_00001': WindowsPath('submissions/your_team_name/generated/sample_00001.png'), 'sample_00002': WindowsPath('submissions/your_team_name/generated/sample_00002.png'), 'sample_00003': WindowsPath('submissions/your_team_name/generated/sample_00003.png'), 'sample_00004': WindowsPath('submissions/your_team_name/generated/sample_00004.png'), 'sample_00005': WindowsPath('submissions/your_team_name/generated/sample_00005.png'), 'sample_00006': WindowsPath('submissions/your_team_name/generated/sample_00006.png'), 'sample_00007': WindowsPath('submissions/your_team_name/generated/sample_00007.png'), 'sample_00008': WindowsPath('submissions/your_team_name/generated/sample_00008.png')}, 'real': {'sample_00000': WindowsPath('submiss

C:\Users\levia\Documents\testing\ISU-Challenge\Starterkit-ICSE-2027\venv\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
C:\Users\levia\Documents\testing\ISU-Challenge\Starterkit-ICSE-2027\venv\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


    Distribution features real: batch 1/1
    Distribution features candidate: batch 1/1
    Distribution features real: batch 1/1
    Distribution features candidate: batch 1/1
Public realism metrics complete.


## 2. 🔍 Failure detection

This section queries categorical in-car scene features with a selectable deterministic VLM baseline. Set `FAILURE_MODEL` in the code cell below to choose between:

- **`moondream`** — the lightweight option. It requires fewer resources and is intended to run on CPU, making it suitable for local smoke tests and machines without a GPU.
- **`qwen`** — Qwen2.5-VL-3B-Instruct. It generally requires more memory and compute, but provides a stronger multimodal baseline when suitable hardware is available.

For each evaluated stem, the selected model runs on the generated image and its matched real reference image (`reference/<stem>_real.jpg`), then compares the predicted feature values. Because only the first 60 scenes (`sample_00000` to `sample_00059`) have a real reference image, failure detection runs on those 60 scenes only. The label JSON files determine which public feature keys are evaluated. A scene is counted as a failure when at least one evaluated feature differs between the generated and real-image predictions.

The report records per-feature matches, raw model answers, inference time, the selected model configuration, the number of failures, the number of executed scenes, and the overall failure rate:

`failure_rate = failures / executed`

This is a behavioural comparison rather than a direct image-quality metric. The public model is a baseline; the designated industrial SUT is used for final evaluation when required.

In [13]:
# Choose 'moondream' for the lightweight CPU-friendly baseline or
# 'qwen' for the stronger Qwen2.5-VL-3B-Instruct baseline.
FAILURE_MODEL = 'moondream'

import image_metrics_utils
importlib.reload(image_metrics_utils)
importlib.reload(track_a_metrics)

evaluate_failure_rate = track_a_metrics.evaluate_failure_rate

# YOU CAN USE FOR EXPERIMENTATION ANY MODEL; FOR RANKING PLS USE QWEN25
# FOR FINAL EVALUATION WE WILL USE AN INDUSTRIAL PROTOTYPE

if FAILURE_MODEL == 'moondream':
    analyzer = MoondreamAnalyzer(revision='2025-06-21', device='cuda:0', seed=0)
elif FAILURE_MODEL == 'qwen':
    analyzer = Qwen25VLAnalyzer(model_name='Qwen/Qwen2.5-VL-3B-Instruct')
else:
    raise ValueError("FAILURE_MODEL must be either 'moondream' or 'qwen'.")

print(f'Using failure-detection model: {FAILURE_MODEL}')
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: {FAILURE_MODEL} failure evaluation for {name}')
    failure_report = evaluate_failure_rate(folders, analyzer)
    reports[name]['failure_raw'] = failure_report
    failure_rate = failure_report['failure_rate']
    failure_text = f'{failure_rate:.3f}' if failure_rate is not None else 'N/A'
    print(
        f'  Failure rate: {failure_text} '
        f"({failure_report['failures']}/{failure_report['executed']})"
    )
    save_report(reports[name], OUTPUT / f'{name}.json')
del analyzer
gc.collect()
print(f'{FAILURE_MODEL} failure-rate evaluation complete.')

Using failure-detection model: moondream
Contestant 1/1: moondream failure evaluation for your_team_name
  MoondreamAnalyzer generated-vs-real matching: 1/9 - sample_00000
  MoondreamAnalyzer generated-vs-real matching: 2/9 - sample_00001
  MoondreamAnalyzer generated-vs-real matching: 3/9 - sample_00002
  MoondreamAnalyzer generated-vs-real matching: 4/9 - sample_00003
  MoondreamAnalyzer generated-vs-real matching: 5/9 - sample_00004
  MoondreamAnalyzer generated-vs-real matching: 6/9 - sample_00005
  MoondreamAnalyzer generated-vs-real matching: 7/9 - sample_00006
  MoondreamAnalyzer generated-vs-real matching: 8/9 - sample_00007
  MoondreamAnalyzer generated-vs-real matching: 9/9 - sample_00008
  Failure rate: 0.889 (8/9)
moondream failure-rate evaluation complete.


## 3. 📋 Final report and failure diversity

This section assembles the previously computed SAM, image-realism, and failure-detection results into one final report for each contestant. It does not remove scenes or filter the raw metric sections.

It then measures diversity among the scenes flagged as failures by the selected failure-detection model (`FAILURE_MODEL`):

- **Feature diversity:** computes the mean pairwise Hamming distance across the categorical label values of failing scenes. A value of `0` means the failing scenes have identical evaluated feature values; larger values indicate more varied failure cases. The report also records the number of failing scenes and the number of unique values per feature.
- **CLIP visual diversity:** computes the mean pairwise cosine distance between CLIP embeddings of the generated images for failing scenes. This is a visual-diversity measure and is `0` when fewer than two failing scenes are available.

The final report is written to `track_a_evaluation_output/<contestant>.json` and retains the raw per-image results, failure details, realism metrics, and diversity summaries.

In [14]:
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: diversity for {name}')
    labels = load_labels(folders['labels'])
    reports[name]['final'] = build_final_report(reports[name], labels)
    failing_stems = [row['image'] for row in reports[name]['final']['failure']['images'] if row['failure']]
    reports[name]['final']['clip_visual_diversity'] = clip_visual_diversity(folders['generated'], failing_stems)
    save_report(reports[name], OUTPUT / f'{name}.json')
print('Diversity evaluation complete.')

Contestant 1/1: diversity for your_team_name
Diversity evaluation complete.


## 4. ⏱️ Efficiency

This section reports the translation runtime for each contestant using organizer-measured `generation_times.json`. The file must contain the standardized total generation time and may include the hardware used for measurement.

For each contestant, the evaluator records:

- **Total generation time:** the measured time for generating the submission.
- **Seconds per scene:** total generation time divided by the number of generated images (all scenes, not only the 60 with a real reference image).
- **Hardware:** the hardware metadata stored in `generation_times.json`.

If standardized timing is unavailable, efficiency is marked as unavailable and is not converted into an estimated value. These measurements are kept in each contestant report; the organizer-only notebook performs the cross-contestant Pareto ranking separately.

In [15]:
for index, (name, folders) in enumerate(contestants.items(), 1):
    # Divide by all generated scenes, not only the 60 scenes used for failure detection.
    generated_scenes = len(list_images(folders['generated']))
    # The participant starter writes the timing file to submissions/<team>/generation_times.json.
    reports[name]['efficiency'] = efficiency(folders['generated'].parent, generated_scenes)
    timing = reports[name]['efficiency'].get('seconds_per_scene')
    timing_text = f'{timing:.3f} s/scene' if timing is not None else 'unavailable'
    print(f'Contestant {index}/{len(contestants)}: {name} - {timing_text}')
    save_report(reports[name], OUTPUT / f'{name}.json')

print('Efficiency reports complete. Run the organizer-only Pareto ranking notebook to rank contestants.')

Contestant 1/1: your_team_name - 0.067 s/scene
Efficiency reports complete. Run the organizer-only Pareto ranking notebook to rank contestants.


## 5. 📄 Readable saved results

This cell reads the saved contestant reports and prints a compact summary. It can be run independently after the evaluation cells. The failure-rate label uses the selected `FAILURE_MODEL`; the organizer-only ranking notebook reads these saved reports to create the Pareto leaderboard.

In [16]:
for path in sorted(OUTPUT.glob('*.json')):
    if path.name == 'leaderboard.json':
        continue
    report = json.loads(path.read_text())
    print(f"\n{report['contestant']}")
    final = report.get('final')
    failure_model = report.get('failure_raw', {}).get('inference', {}).get('model', 'selected model')
    if final is None:
        raw = report.get('failure_raw', {})
        print(f"  Final report not assembled yet; {failure_model} scenes: {raw.get('executed', 0)}")
        continue
    failure = final['failure']
    failure_rate = failure.get('failure_rate')
    failure_text = f'{failure_rate:.3f}' if failure_rate is not None else 'N/A'
    print(f"  {failure_model} disagreement rate: {failure_text} ({failure['failures']}/{failure['executed']})")
    generated = final['realism']['generated_vs_real']['summary']
    simulated = final['realism']['simulated_vs_real']['summary']
    generated_ssim = generated.get('ssim')
    simulated_ssim = simulated.get('ssim')
    generated_text = f'{generated_ssim:.3f}' if generated_ssim is not None else 'N/A'
    simulated_text = f'{simulated_ssim:.3f}' if simulated_ssim is not None else 'N/A'
    print(f"  SSIM to real: generated {generated_text}, simulated {simulated_text}")
    sam = report.get('sam', {})
    sim_sam = sam.get('simulated_vs_generated', {}).get('summary', {}).get('partition', {})
    print(f"  SAM boundary F1: simulated_vs_generated {sim_sam.get('boundary_f1', float('nan')):.3f}")
    print(f"  Feature diversity: {final['feature_diversity']['mean_pairwise_hamming']:.3f}")
    timing = report.get('efficiency', {})
    seconds_per_scene = timing.get('seconds_per_scene')
    timing_text = f'{seconds_per_scene:.3f} s/scene' if seconds_per_scene is not None else 'N/A'
    print(f"  Generation time: {timing_text}")


your_team_name
  vikhyatk/moondream2 disagreement rate: 0.889 (8/9)
  SSIM to real: generated 0.331, simulated 0.467
  SAM boundary F1: simulated_vs_generated 0.860
  Feature diversity: 0.313
  Generation time: 0.067 s/scene
